***

*Course:* [Math 535](https://people.math.wisc.edu/~roch/mmids/) - Mathematical Methods in Data Science (MMiDS)  
*Chapter:* 3-Singular value decomposition   
*Author:* [Sebastien Roch](https://people.math.wisc.edu/~roch/), Department of Mathematics, University of Wisconsin-Madison  
*Updated:* Jan 27, 2024   
*Copyright:* &copy; 2024 Sebastien Roch

***

In [ ]:
# FOR e-BOOK ONLY
import os, sys
sys.path.insert(0, os.path.abspath('../../utils')) # use directory to mmids.py

In [ ]:
# PYTHON 3
import numpy as np
from numpy import linalg as LA
import matplotlib.pyplot as plt
import pandas as pd
import mmids
seed = 535
rng = np.random.default_rng(seed)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# FOR TeX ONLY
#plt.rcParams['figure.figsize'] = [4.00, 2.25] # for high-def figs
#plt.rcParams['figure.dpi'] = 200 # for high-def figs

$\newcommand{\horz}{\rule[.5ex]{2.5ex}{0.5pt}}$

## Application to dimensionality reduction

We discuss an application to principal components analysis and revisit our genetic dataset from ealier in the chapter.

### Principal components analysis

Principal components analysis (PCA) is a commonly used dimensionality reduction approach that is mathematically equivalent closely related to what we described in the previous sections. We formalize the connection.

*The data matrix:* In PCA we are given $n$ data points $\mathbf{x}_1,\ldots,\mathbf{x}_n \in \mathbb{R}^p$ with $p$ features (i.e., coordinates). We denote the components of $\mathbf{x}_i$ as $(x_{i1},\ldots,x_{ip})^T$. As usual, we stack them up into a matrix $X$ whose $i$-th row is $\mathbf{x}_i^T$. 

The first step of PCA is to center the data, i.e., we assume that

$$
\frac{1}{n} \sum_{i=1}^n x_{ij} = 0, \qquad \forall j=1,\ldots,p
$$

Put differently, the empirical mean of each column is $0$. Quoting [Wikipedia](https://en.wikipedia.org/wiki/Principal_component_analysis#Further_considerations) (and this will become clearer below):

> Mean subtraction (a.k.a. "mean centering") is necessary for performing classical PCA to ensure that the first principal component describes the direction of maximum variance. If mean subtraction is not performed, the first principal component might instead correspond more or less to the mean of the data. A mean of zero is needed for finding a basis that minimizes the mean square error of the approximation of the data.

An optional step is to divide each column by the square root of its [sample variance](https://en.wikipedia.org/wiki/Variance#Unbiased_sample_variance), i.e., assume that

$$
\frac{1}{n-1} \sum_{i=1}^n x_{ij}^2 = 1, \qquad \forall j=1,\ldots,p.
$$

As we mentioned in a previous chapter, this is particularly important when the features are measured in different units to ensure that their variability can be meaningfully compared.

*The first principal component:* The first principal component is the linear combination of the features

$$
t_{i1}
= \phi_{11} x_{i1} + \cdots + \phi_{p1} x_{ip}
$$

with largest sample variance. For this to make sense, we need to constrain the $\phi_{j1}$'s. Specifically, we require

$$
\sum_{j=1}^p \phi_{j1}^2 = 1.
$$

The $\phi_{j1}$'s are referred to as the *loadings* and the $t_{i1}$'s are referred to as the *scores*.

Formally, we seek to solve

$$
\max\left\{\frac{1}{n-1} \sum_{i=1}^n \left(\sum_{j=1}^p \phi_{j1} x_{ij}\right)^2\ :\ \sum_{j=1}^p \phi_{j1}^2 = 1\right\},
$$

where we used the fact that the $t_{i1}$'s are centered

\begin{align*}
\frac{1}{n} \sum_{i=1}^n t_{i1}
&= \frac{1}{n} \sum_{i=1}^n [\phi_{11} x_{i1} + \cdots + \phi_{p1} x_{ip}]\\
&= \phi_{11} \frac{1}{n} \sum_{i=1}^n  x_{i1} + \cdots + \phi_{p1} \frac{1}{n} \sum_{i=1}^n  x_{ip}\\
&= 0,
\end{align*}

to compute their sample variance as the mean of their square

$$
\frac{1}{n-1}\sum_{i=1}^n t_{i1}^2 
= \frac{1}{n-1} \sum_{i=1}^n \left(\sum_{j=1}^p \phi_{j1} x_{ij}\right)^2.
$$

Let $\boldsymbol{\phi}_1 = (\phi_{11},\ldots,\phi_{p1})^T$ and $\mathbf{t}_1 = (t_{11},\ldots,t_{n1})^T$. Then for all $i$

$$
t_{i1} = \mathbf{x}_i^T \boldsymbol{\phi}_1,
$$

or in vector form

$$
\mathbf{t}_1 = X \boldsymbol{\phi}_1.
$$

Also

$$
\frac{1}{n-1}\sum_{i=1}^n t_{i1}^2 
= \frac{1}{n-1} \|\mathbf{t}_1\|^2
= \frac{1}{n-1} \|X \boldsymbol{\phi}_1\|^2.
$$

Rewriting the maximization problem above in matrix form, 

$$
\max\left\{\frac{1}{n-1} \|X \boldsymbol{\phi}_1\|^2 : \|\boldsymbol{\phi}_1\|^2 = 1\right\},
$$

we see that we have already encountered this problem (up to the factor of $1/(n-1)$ which does not affect the solution). The solution is to take $\boldsymbol{\phi}_1$ to be the top right singular vector of $X$.

*The second principal component:* The second principal component is the linear combination of the features

$$
t_{i2}
= \phi_{12} x_{i1} + \cdots + \phi_{p2} x_{ip}
$$

with largest sample variance that is also uncorrelated with the first principal component, in the sense that

$$
\frac{1}{n-1} \sum_{i=1}^n t_{i1} t_{i2} = 0.
$$

The next lemma shows how to deal with this condition. Again, we also require

$$
\sum_{j=1}^p \phi_{j2}^2 = 1.
$$

As before, let $\boldsymbol{\phi}_2 = (\phi_{12},\ldots,\phi_{p2})^T$ and $\mathbf{t}_2 = (t_{12},\ldots,t_{n2})^T$.

**LEMMA** **(Uncorrelated Principal Components)** Assume $X \neq \mathbf{0}$. Let $t_{i1}$, $t_{i2}$, $\boldsymbol{\phi}_1$, $\boldsymbol{\phi}_2$ be as above. Then

$$
\frac{1}{n-1} \sum_{i=1}^n t_{i1} t_{i2} = 0
$$

holds if and only if

$$
\langle \boldsymbol{\phi}_1, \boldsymbol{\phi}_2 \rangle = 0.
$$

$\flat$

*Proof:* The condition 

$$
\frac{1}{n-1} \sum_{i=1}^n t_{i1} t_{i2} = 0
$$

is equivalent to

$$
\langle \mathbf{t}_1, \mathbf{t}_2 \rangle = 0,
$$

where we dropped the $1/n$ factor as it does not play any role. Using that $\mathbf{t}_1 = X \boldsymbol{\phi}_1$, and similarly, $\mathbf{t}_2 = X \boldsymbol{\phi}_2$, this is in turn equivalent to

$$
\langle X \boldsymbol{\phi}_1, X \boldsymbol{\phi}_2 \rangle = 0.
$$

Because $\boldsymbol{\phi}_1$ can be chosen as a top right singular vector in an SVD of $X$, it follows from the *SVD Relations* that
$X^T X \boldsymbol{\phi}_1 = \sigma_1^2 \boldsymbol{\phi}_1$, where $\sigma_1$ is the singular value associated to $\boldsymbol{\phi}_1$. Since $X \neq 0$, $\sigma_1 > 0$. Plugging this in the inner product on the left hand side above, we get

\begin{align*}
\langle X \boldsymbol{\phi}_1, X \boldsymbol{\phi}_2 \rangle
&= \langle X \boldsymbol{\phi}_2, X \boldsymbol{\phi}_1 \rangle\\
&= (X \boldsymbol{\phi}_2)^T (X \boldsymbol{\phi}_1)\\
&= \boldsymbol{\phi}_2^T X^T X \boldsymbol{\phi}_1\\
&= \boldsymbol{\phi}_2^T (\sigma_1 \boldsymbol{\phi}_1)\\
&= \langle \boldsymbol{\phi}_2, \sigma_1 \boldsymbol{\phi}_1 \rangle\\
&= \sigma_1 \langle \boldsymbol{\phi}_1, \boldsymbol{\phi}_2 \rangle.
\end{align*}

Because $\sigma_1 \neq 0$, this is $0$ if and only if $\langle \boldsymbol{\phi}_1, \boldsymbol{\phi}_2 \rangle = 0$. $\square$

As a result, we can write the maximization problem for the second principal component in matrix form as

$$
\max\left\{\frac{1}{n-1} \|X \boldsymbol{\phi}_2\|^2 : \|\boldsymbol{\phi}_2\|^2 = 1, \langle \boldsymbol{\phi}_1, \boldsymbol{\phi}_2 \rangle = 0\right\}.
$$

Again, we see that we have encountered this problem before. The solution is to take $\boldsymbol{\phi}_2$ to be a second right singular vector in an SVD of $X$.

*Further principal components:* We can proceed in a similar fashion and define further principal components.

To quote [Wikipedia](https://en.wikipedia.org/wiki/Principal_component_analysis#Further_considerations):

> PCA essentially rotates the set of points around their mean in order to align with the principal components. This moves as much of the variance as possible (using an orthogonal transformation) into the first few dimensions. The values in the remaining dimensions, therefore, tend to be small and may be dropped with minimal loss of information (see below). PCA is often used in this manner for dimensionality reduction. PCA has the distinction of being the optimal orthogonal transformation for keeping the subspace that has largest "variance" (as defined above).

Formally, let 

$$
X = U \Sigma V^T
$$

be the SVD of the data matrix $X$. The principal component transformation, truncated at the $\ell$-th component, is

$$
T = X V_{(\ell)}, 
$$

where $T$ is the matrix whose colums are the vectors $\mathbf{t}_1,\ldots,\mathbf{t}_\ell$. Recall that $V_{(\ell)}$ is the matrix made of the first $k$ columns of $V$. 

Then, using the orthonormality of the right singular vectors,

$$
T 
= U \Sigma V^T V_{(\ell)}
= U \Sigma \begin{bmatrix} I_{\ell \times \ell}\\
\mathbf{0}_{(p-\ell)\times \ell}\end{bmatrix}
= U \begin{bmatrix}\Sigma_{(\ell)}\\ \mathbf{0}_{(p-\ell)\times \ell}\end{bmatrix}
= U_{(\ell)} \Sigma_{(\ell)}.
$$

Put differently, the vector $\mathbf{t}_i$ is the left singular vector $\mathbf{u}_i$ scaled by the corresponding singular value $\sigma_i$.

**NUMERICAL CORNER:** Having established a formal connection between PCA and SVD, we implement PCA using our SVD algorithm (in [mmids.py](https://raw.githubusercontent.com/MMiDS-textbook/MMiDS-textbook.github.io/main/utils/mmids.py)). We perform mean centering (now is the time to read that quote about the importance of mean centering again), but not the optional standardization. We use the fact that, in Numpy, subtracting a matrix by a vector whose dimension matches the number of columns performs row-wise subtraction.

In [ ]:
def pca(X, l, maxiter=100):
    mean = np.mean(X, axis=0)  # Compute mean of each column
    Y = X - mean # Mean center each column
    U, S, V = mmids.svd(Y, l, maxiter=maxiter)
    return U[:,:l] @ np.diag(S[:l])

We apply it to the Gaussian Mixture Model. 

In [ ]:
d, n, w = 1000, 100, 3.
X = mmids.two_mixed_clusters(d, n, w)

In [ ]:
T = pca(X, 2)

Plotting the result, we see that PCA does succeed in finding the main direction of variation, with the first principal component aligning well with the first coordinate.

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111,aspect='equal')
ax.scatter(T[:,0], T[:,1], s=10)
plt.show()

Note however that the first two principal components (especially the second one) in fact "capture more noise" than what can be seen in the first two coordinates, a form of overfitting. 

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111,aspect='equal')
ax.scatter(X[:,0], X[:,1], s=10)
plt.show()

**TRY IT!** Compute the first two right singular vectors $\mathbf{v}_1$ and $\mathbf{v}_2$ of $X$ after mean centering. Do they align well with the first and second standard basis vectors $\mathbf{e}_1$ and $\mathbf{e}_2$? Why or why not? ([Open in Colab](https://colab.research.google.com/github/MMiDS-textbook/MMiDS-textbook.github.io/blob/main/just_the_code/roch_mmids_chap_svd_notebook.ipynb))

$\unlhd$

### Back to our genetic dataset

We return to our motivating example. We apply PCA to our genetic dataset.

**Figure:** Viruses (*Credit:* Made with [Midjourney](https://www.midjourney.com/))

![Viruses](./figs/3D_visualization_of_virus-small.png)

$\bowtie$

We load the dataset again and examine its first rows.

In [ ]:
df = pd.read_csv('h3n2-snp.csv')

In [ ]:
df.head()

Recall that it contains $1642$ strains and lives in a $318$-dimensional space. 

In [ ]:
df.shape

Our goal is to find a "good" low-dimensional representation of the data. Ten dimensions will do here. We use PCA. 

In [ ]:
A = df[[df.columns[i] for i in range(1,len(df.columns))]].to_numpy()

In [ ]:
n_dims = 10
T = pca(A, n_dims)

We plot the first two principal components, and see what appears to be some potential structure. 

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111,aspect='equal')
ax.scatter(T[:,0], T[:,1], s=10)
plt.show()

There seems to be some reasonably well-defined clusters in this projection. We use $k$-means to identiy clusters. We take advantage of the implementation in scikit-learn, [sklearn.cluster.KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html). By default, it finds $8$ clusters. The clusters can be extracted from the attribute `labels_`.

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
n_clusters = 8
kmeans = KMeans(n_clusters=n_clusters, init='k-means++', n_init=10).fit(T)

In [ ]:
assign = kmeans.labels_

To further reveal the structure, we look at our the clusters spread out over the years. That information is in a separate file. 

In [ ]:
dfoth = pd.read_csv('h3n2-other.csv')
dfoth.head()

In [ ]:
year = dfoth['year'].to_numpy()

For each cluster, we plot how many of its data points come from a specific year. Each cluster has a different color.

In [ ]:
fig, ax = plt.subplots()

for i in range(n_clusters):
    unique, counts = np.unique(year[assign == i], return_counts=True)
    ax.bar(unique, counts, label=i)

ax.set(xlim=(2001, 2007), xticks=np.arange(2002, 2007))
ax.legend()
plt.show()

Remarkably, we see that each cluster comes mostly from one year or two consecutive ones. In other words, the clustering in this low-dimensional projection captures some true underlying structure that is not explicitly in the genetic data on which it is computed.

Going back to the first two principal components, we color the points on the scatterplot by year. (We use [`legend_elements()`](https://matplotlib.org/stable/api/collections_api.html#matplotlib.collections.PathCollection.legend_elements) for automatic legend creation.) 

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111, aspect='equal')
scatter = ax.scatter(T[:,0], T[:,1], s=10,  c=year, label=year)
plt.legend(*scatter.legend_elements())
plt.show()

To some extent, one can "see" the virus evolving from year to year. The $x$-axis in particular seems to correlate strongly with the year, in the sense that samples from later years tend to be towards one side of the plot. 

To further quantify this observation, we use [numpy.corrcoef](https://numpy.org/doc/stable/reference/generated/numpy.corrcoef.html) to compute the correlation coefficients between the year and the first $10$ principal components. 

In [ ]:
corr = np.zeros(n_dims)
for i in range(n_dims):
    corr[i] = np.corrcoef(np.stack((T[:,i], year)))[0,1]

print(corr)

Indeed, we see that the first three or four principal components correlate well with the year.

Using [related techniques](https://bmcgenet.biomedcentral.com/articles/10.1186/1471-2156-11-94/figures/8), one can also identify which mutations distinguish different epidemics (i.e., years). 

**LEARNING BY CHATTING** Ask your favorite AI chatbots about the difference between principal components analysis (PCA) and linear discriminant analysis (LDA). 